<a href="https://colab.research.google.com/github/aandreeaion/Analysing_Data_A3/blob/main/Analysing_Data_Assignment3_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — Genre Classification of Song Lyrics with Ollama

This notebook implements:

1. **Zero-shot prompting**
2. **Few-shot prompting**

It evaluates both strategies on the provided `genreLyrics_train.csv` and `genreLyrics_test.csv` files using:

- Precision
- Recall
- F1 score

It also reports **per-genre performance** and stores the raw model output so it can be inspected before scoring.

## 1. Setting up

Note: I recommend running the code in Google Colab or another environment where Ollama can be installed and started.

In [1]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
!cat ollama.log

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new 

In [6]:
# pulling a compact model
!ollama pull llama3.2:3b
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


## 2. Importing

In [3]:
!pip install -q ollama pandas scikit-learn tqdm

In [4]:
import re
from pathlib import Path

import ollama
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, precision_recall_fscore_support

## 3. Loading the data

Before running the next code cell in Google Colab, upload the following files to the _**Files**_ panel on the left side:

- `genreLyrics_train.csv`
- `genreLyrics_test.csv`

The training data is used to construct few-shot examples, while the test data is used for final evaluation.
Use this sentence in the markdown cell above it:

I inspect the shape, columns, and first rows of the datasets to confirm that the files loaded correctly and contain the expected genre and lyrics fields.


In [7]:
train_path = Path("genreLyrics_train.csv")
test_path = Path("genreLyrics_test.csv")

train_df = pd.read_csv(train_path, sep="\t")
test_df = pd.read_csv(test_path, sep="\t")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain columns:", train_df.columns.tolist())

display(train_df.head(3))

Train shape: (5000, 3)
Test shape: (1500, 3)

Train columns: ['Unnamed: 0', 'genre', 'lyrics']


,Unnamed: 0,genre,lyrics
0,263935,Rock,"I knew a man, called him Sandy Cane\nFew folks..."
1,64235,Country,(Gary Harrison - Kent Robbins - Gene Miller)\n...
2,197695,Rock,You think its right when you see my reaction\n...


In [8]:
print(train_df.dtypes)
print()
print(test_df.dtypes)

Unnamed: 0     int64
genre         object
lyrics        object
dtype: object

Unnamed: 0     int64
genre         object
lyrics        object
dtype: object


## 4. Defining the labels and inspecting class distribution

In [13]:
GENRES = sorted(train_df["genre"].unique().tolist())

print("Genres:")
print(GENRES)
print("\nTraining set genre counts:")
print(train_df["genre"].value_counts())

Genres:
['Country', 'Electronic', 'Folk', 'Hip-Hop', 'Indie', 'Jazz', 'Metal', 'Other', 'Pop', 'R&B', 'Rock']

Training set genre counts:
genre
Rock          2213
Pop            843
Hip-Hop        518
Metal          498
Country        302
Jazz           166
Electronic     156
Other          111
Indie           74
R&B             67
Folk            52
Name: count, dtype: int64


## 5. Creating inspection samples

I create smaller samples to check whether the prompts and raw model outputs look correct before running the model on the full test set.

In [14]:
# small balanced sample: 2 random examples per genre from the test set
sample_test_df = (
    test_df.groupby("genre", group_keys=False)
    .sample(2, random_state=42)
    .reset_index(drop=True)
)

# medium balanced sample: 10 random examples per genre from the test set
medium_test_df = (
    test_df.groupby("genre", group_keys=False)
    .sample(10, random_state=42)
    .reset_index(drop=True)
)

print("Small sample shape:", sample_test_df.shape)
print(sample_test_df["genre"].value_counts())

print("\nMedium sample shape:", medium_test_df.shape)
print(medium_test_df["genre"].value_counts())

Small sample shape: (22, 3)
genre
Country       2
Electronic    2
Folk          2
Hip-Hop       2
Indie         2
Jazz          2
Metal         2
Other         2
Pop           2
R&B           2
Rock          2
Name: count, dtype: int64

Medium sample shape: (110, 3)
genre
Country       10
Electronic    10
Folk          10
Hip-Hop       10
Indie         10
Jazz          10
Metal         10
Other         10
Pop           10
R&B           10
Rock          10
Name: count, dtype: int64


## 6. Building zero-shot and few-shot prompts

In [15]:
def build_zero_shot_prompt(lyrics: str, genres: list[str]) -> str:
    # creating a zero-shot prompt that asks the model to return one label only
    return f"""
You are a music genre classifier.

Choose exactly one genre for the lyrics from this closed set of labels:
{", ".join(genres)}

Important:
- You must answer with exactly one label from the list above.
- Do not invent new labels.
- Do not use related labels such as Rap, Industrial, Christian, Romantic, Soul, Alternative, etc.
- If uncertain, choose the closest label from the list.
- Output only the label.

Lyrics:
\"\"\"
{lyrics}
\"\"\"

Label:
""".strip()


def build_few_shot_prompt(lyrics: str, genres: list[str], examples_df: pd.DataFrame) -> str:
    # creating a few-shot prompt using labeled training examples
    examples_text = []

    for _, row in examples_df.iterrows():
        examples_text.append(
            f"""Lyrics:
\"\"\"
{row['lyrics']}
\"\"\"
Label: {row['genre']}"""
        )

    examples_block = "\n\n".join(examples_text)

    return f"""
You are a music genre classifier.

Choose exactly one genre for the lyrics from this closed set of labels:
{", ".join(genres)}

Important:
- You must answer with exactly one label from the list above.
- Do not invent new labels.
- Do not use related labels such as Rap, Industrial, Christian, Romantic, Soul, Alternative, etc.
- If uncertain, choose the closest label from the list.
- Output only the label.

Here are labeled examples:
{examples_block}

Now classify this new lyric.

Lyrics:
\"\"\"
{lyrics}
\"\"\"

Label:
""".strip()

In [16]:
print("Zero-shot prompt preview:\n")
print(build_zero_shot_prompt(sample_test_df.loc[0, "lyrics"], GENRES)[:1200])

Zero-shot prompt preview:

You are a music genre classifier.

Choose exactly one genre for the lyrics from this closed set of labels:
Country, Electronic, Folk, Hip-Hop, Indie, Jazz, Metal, Other, Pop, R&B, Rock

Important:
- You must answer with exactly one label from the list above.
- Do not invent new labels.
- Do not use related labels such as Rap, Industrial, Christian, Romantic, Soul, Alternative, etc.
- If uncertain, choose the closest label from the list.
- Output only the label.

Lyrics:
"""
Woody walked out of the cotton field
In his head he had words that his lips concealed
An old airstrip where the weeds are growin'
A burned out bomber with the cockpit glowin'
With the key from a room to the Skyview Motel
He scratched words in the wing as a falling star fell
The moon sees you, the moon sees me
The moon sees who I want to see
I'll see you soon
Down in the light of the melon moon
Down in the light of the melon moon
On the other side of town 'n another side of life
Stella hold

## 7. Creating few-shot examples from the training set

For few-shot prompting, I select one random labeled example per genre from the training data.  
These examples are included in the prompt to guide the model toward the predefined genre labels.

In [17]:
# random_state=42so that the same random examples are selected each time the code runs
few_shot_examples = (
    train_df.groupby("genre", group_keys=False)
    .sample(1, random_state=42)[["genre", "lyrics"]]
    .reset_index(drop=True)
)

print("Few-shot examples shape:", few_shot_examples.shape)
display(few_shot_examples)

Few-shot examples shape: (11, 2)


,genre,lyrics
0,Country,"Take me back to Tulsa, I'm too young to marry\..."
1,Electronic,(Verse 1)\nLike the legend of the phoenix\nAll...
2,Folk,"""Queensland Whalers""\n-Eric Bogle\nI've sailed..."
3,Hip-Hop,Sorry is not enough\nAfter all the things I've...
4,Indie,"Scarecrow, you ruined me.\nNow I've caught my ..."
5,Jazz,"Well, what do you know, he smiled at me in my ..."
6,Metal,"""Death Don't Want You""\n-Blaze-\nIt ain't no p..."
7,Other,"Ay, huh, look\nOh you thought it was a game hu..."
8,Pop,You know Christmas was made for the chi'ren\nD...
9,R&B,"A prayer, just what would I say?\nSpeak of tho..."


In [18]:
print("Few-shot prompt preview:\n")
print(build_few_shot_prompt(sample_test_df.loc[0, "lyrics"], GENRES, few_shot_examples)[:2000])

Few-shot prompt preview:

You are a music genre classifier.

Choose exactly one genre for the lyrics from this closed set of labels:
Country, Electronic, Folk, Hip-Hop, Indie, Jazz, Metal, Other, Pop, R&B, Rock

Important:
- You must answer with exactly one label from the list above.
- Do not invent new labels.
- Do not use related labels such as Rap, Industrial, Christian, Romantic, Soul, Alternative, etc.
- If uncertain, choose the closest label from the list.
- Output only the label.

Here are labeled examples:
Lyrics:
"""
Take me back to Tulsa, I'm too young to marry
Take me back to Tulsa, I'm too young to marry
You see that girl with the red dress on,
Some folks call her Dinah
Stoled my heart away from me
Way down in Louisiana
Take me back to Tulsa, I'm too young to marry
Take me back to Tulsa, I'm too young to marry
The big bee sucks the blossom
And the little bee makes the honey
Poor man throws the cotton
And the rich man makes the money
Take me back to Tulsa, I'm too young to m

## 8. Normalizing model outputs before scoring

LLMs do not always return a clean genre label. Sometimes they add extra words, punctuation, or formatting.

e.g.: The genre is Rock/ I think this is Jazz.

I define a helper function cleans those outputs and maps them back to the predefined set of genre labels.

In [19]:
def normalize_text_for_match(text: str) -> str:
    # lowercasing and simplifying text so it can be matched more reliably
    text = text.lower().strip()
    text = text.replace("&", "and")
    text = text.replace("-", " ")
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_prediction(text: str, valid_labels: list[str]) -> str:
    # returning a valid genre label if one can be found, otherwise UNKNOWN
    if text is None:
        return "UNKNOWN"

    text = text.strip().replace("```", "").strip()
    first_line = text.split("\n")[0].strip()

    norm_first = normalize_text_for_match(first_line)
    norm_full = normalize_text_for_match(text)

    label_map = {normalize_text_for_match(label): label for label in valid_labels}

    # exact match on the first line
    if norm_first in label_map:
        return label_map[norm_first]

    # exact match anywhere in the full response
    if norm_full in label_map:
        return label_map[norm_full]

    # searching for any valid label as a phrase inside the response
    for normalized_label, original_label in label_map.items():
        pattern = rf"\b{re.escape(normalized_label)}\b"
        if re.search(pattern, norm_first) or re.search(pattern, norm_full):
            return original_label

    return "UNKNOWN"

In [20]:
# checks
print(normalize_prediction("Hip-Hop", GENRES))
print(normalize_prediction("The genre is Rock", GENRES))
print(normalize_prediction("I think this is probably Jazz.", GENRES))
print(normalize_prediction("not sure", GENRES))

Hip-Hop
Rock
Jazz
UNKNOWN


## 9. Functions for zero-shot and few-shot classification

In this section, I define the functions that send song lyrics to the model, collect the raw response, and return both the raw and cleaned prediction for zero-shot and few-shot prompting.

In [21]:
model_name = "llama3.2:3b"


def classify_lyrics_zero_shot(lyrics: str, model: str = model_name) -> tuple[str, str]:
    # classifying one song's lyrics using zero-shot prompting
    prompt = build_zero_shot_prompt(lyrics, GENRES)

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )

    raw_output = response["message"]["content"]
    pred = normalize_prediction(raw_output, GENRES)
    return raw_output, pred


def classify_lyrics_few_shot(
    lyrics: str,
    examples_df: pd.DataFrame,
    model: str = model_name,
) -> tuple[str, str]:
    # classifying one song's lyrics using few-shot prompting
    prompt = build_few_shot_prompt(lyrics, GENRES, examples_df)

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )

    raw_output = response["message"]["content"]
    pred = normalize_prediction(raw_output, GENRES)
    return raw_output, pred

## 10. Inspecting raw outputs on a small sample before scoring

Here I check a few example predictions so I can see exactly what the model is returning before evaluating the full dataset.

In [22]:
for i in range(5):
    lyrics = sample_test_df.loc[i, "lyrics"]
    true_label = sample_test_df.loc[i, "genre"]

    raw_output, pred = classify_lyrics_zero_shot(lyrics)

    print(f"Example {i}")
    print("True label:", true_label)
    print("Raw output:", raw_output)
    print("Normalized prediction:", pred)
    print("-" * 80)

Example 0
True label: Country
Raw output: Pop
Normalized prediction: Pop
--------------------------------------------------------------------------------
Example 1
True label: Country
Raw output: Indie
Normalized prediction: Indie
--------------------------------------------------------------------------------
Example 2
True label: Electronic
Raw output: Folk
Normalized prediction: Folk
--------------------------------------------------------------------------------
Example 3
True label: Electronic
Raw output: Country.
Normalized prediction: Country
--------------------------------------------------------------------------------
Example 4
True label: Folk
Raw output: Rock
Normalized prediction: Rock
--------------------------------------------------------------------------------


In [23]:
for i in range(5):
    lyrics = sample_test_df.loc[i, "lyrics"]
    true_label = sample_test_df.loc[i, "genre"]

    raw_output, pred = classify_lyrics_few_shot(lyrics, few_shot_examples)

    print(f"Example {i}")
    print("True label:", true_label)
    print("Raw output:", raw_output)
    print("Normalized prediction:", pred)
    print("-" * 80)

Example 0
True label: Country
Raw output: Based on the lyrics provided, I would classify this song as Rock. The themes, imagery, and language used are consistent with the genre, and the style of the writing suggests a rock or folk-rock band. The lyrical focus on storytelling, poetic imagery, and atmospheric description is also characteristic of many rock songs from the 1970s to present day.

Specifically, the lyrics seem to evoke a sense of nostalgia, longing, and melancholy, which are common themes in rock music, particularly in the genres of folk-rock, country-rock, and Americana. The use of vivid imagery, such as "the moon sees you" and "the light of the melon moon," suggests a focus on atmosphere and mood, which is typical of many rock songs.

If I had to narrow it down further, I would say that this song could be classified as either:

* Folk-Rock: due to the emphasis on storytelling and poetic imagery
* Soft Rock/ Adult Contemporary: due to the mellow and introspective tone of th

## 11. Running classification experiments

I define a function that runs the zero-shot or few-shot classification process over a dataframe and stores the true labels, raw model outputs, and cleaned predictions.

In [24]:
def run_experiment(
    df: pd.DataFrame,
    strategy: str,
    examples_df: pd.DataFrame | None = None,
    model: str = model_name,
) -> pd.DataFrame:
    # running zero-shot or few-shot classification over a dataframe of lyrics
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        lyrics = row["lyrics"]
        true_label = row["genre"]

        if strategy == "zero_shot":
            raw_output, pred = classify_lyrics_zero_shot(lyrics, model=model)
        elif strategy == "few_shot":
            raw_output, pred = classify_lyrics_few_shot(lyrics, examples_df, model=model)
        else:
            raise ValueError("strategy must be 'zero_shot' or 'few_shot'")

        results.append(
            {
                "true_genre": true_label,
                "raw_output": raw_output,
                "pred_genre": pred,
            }
        )

    return pd.DataFrame(results)

## 12. Evaluation functions

I define functions that calculate and show the evaluation results after the model has made its predictions.

In [25]:
def classification_report_df(results_df: pd.DataFrame, labels: list[str]) -> pd.DataFrame:
    # returning per-genre precision, recall, F1, and support as a dataframe
    p, r, f1, support = precision_recall_fscore_support(
        results_df["true_genre"],
        results_df["pred_genre"],
        labels=labels,
        zero_division=0,
    )

    return pd.DataFrame(
        {
            "genre": labels,
            "precision": p,
            "recall": r,
            "f1": f1,
            "support": support,
        }
    )


def overall_scores_df(results_df: pd.DataFrame, labels: list[str]) -> pd.DataFrame:
    # returning macro and weighted precision/recall/F1 for easier comparison
    rows = []
    for average in ["macro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            results_df["true_genre"],
            results_df["pred_genre"],
            labels=labels,
            average=average,
            zero_division=0,
        )
        rows.append(
            {
                "average": average,
                "precision": p,
                "recall": r,
                "f1": f1,
            }
        )
    return pd.DataFrame(rows)


def print_text_report(results_df: pd.DataFrame, labels: list[str]) -> None:
    # printing the classification report
    print(
        classification_report(
            results_df["true_genre"],
            results_df["pred_genre"],
            labels=labels,
            zero_division=0,
        )
    )

## 13. Evaluating on the small sample

In this section I run both zero-shot and few-shot prompting on a small balanced sample and prints the evaluation results. This checks if the evaluation pipeline works and to get an initial comparison before testing on the large dataset.

In [26]:
zero_shot_sample_df = run_experiment(sample_test_df, strategy="zero_shot")
few_shot_sample_df = run_experiment(
    sample_test_df,
    strategy="few_shot",
    examples_df=few_shot_examples,
)

print("Zero-shot sample report")
print_text_report(zero_shot_sample_df, GENRES)

print("Few-shot sample report")
print_text_report(few_shot_sample_df, GENRES)

100%|██████████| 22/22 [01:47<00:00,  4.91s/it]

Zero-shot sample report
              precision    recall  f1-score   support

     Country       1.00      1.00      1.00         2
  Electronic       0.00      0.00      0.00         2
        Folk       0.00      0.00      0.00         2
     Hip-Hop       0.00      0.00      0.00         2
       Indie       0.00      0.00      0.00         2
        Jazz       0.00      0.00      0.00         2
       Metal       0.25      0.50      0.33         2
       Other       0.00      0.00      0.00         2
         Pop       0.08      0.50      0.14         2
         R&B       0.00      0.00      0.00         2
        Rock       0.00      0.00      0.00         2

    accuracy                           0.18        22
   macro avg       0.12      0.18      0.13        22
weighted avg       0.12      0.18      0.13        22

Few-shot sample report
              precision    recall  f1-score   support

     Country       0.00      0.00      0.00         2
  Electronic       0.50      0.

In [27]:
display(classification_report_df(zero_shot_sample_df, GENRES))
display(classification_report_df(few_shot_sample_df, GENRES))

,genre,precision,recall,f1,support
0,Country,1.000000,1.0,1.000000,2
1,Electronic,0.000000,0.0,0.000000,2
2,Folk,0.000000,0.0,0.000000,2
3,Hip-Hop,0.000000,0.0,0.000000,2
4,Indie,0.000000,0.0,0.000000,2
5,Jazz,0.000000,0.0,0.000000,2
6,Metal,0.250000,0.5,0.333333,2
7,Other,0.000000,0.0,0.000000,2
8,Pop,0.083333,0.5,0.142857,2
9,R&B,0.000000,0.0,0.000000,2


,genre,precision,recall,f1,support
0,Country,0.000000,0.0,0.000000,2
1,Electronic,0.500000,0.5,0.500000,2
2,Folk,0.000000,0.0,0.000000,2
3,Hip-Hop,1.000000,0.5,0.666667,2
4,Indie,0.000000,0.0,0.000000,2
5,Jazz,0.000000,0.0,0.000000,2
6,Metal,0.500000,0.5,0.500000,2
7,Other,0.000000,0.0,0.000000,2
8,Pop,0.000000,0.0,0.000000,2
9,R&B,0.000000,0.0,0.000000,2


## 14. Evaluating on the medium sample

In this section, I  perform the same kind of comparison as in section 13, but on a larger sample.

In [28]:
zero_shot_medium_df = run_experiment(medium_test_df, strategy="zero_shot")
few_shot_medium_df = run_experiment(
    medium_test_df,
    strategy="few_shot",
    examples_df=few_shot_examples,
)

print("Zero-shot medium report")
print_text_report(zero_shot_medium_df, GENRES)

print("Few-shot medium report")
print_text_report(few_shot_medium_df, GENRES)

100%|██████████| 110/110 [09:18<00:00,  5.08s/it]

Zero-shot medium report
              precision    recall  f1-score   support

     Country       0.09      0.10      0.10        10
  Electronic       0.00      0.00      0.00        10
        Folk       0.10      0.10      0.10        10
     Hip-Hop       0.67      0.20      0.31        10
       Indie       0.00      0.00      0.00        10
        Jazz       0.00      0.00      0.00        10
       Metal       0.50      0.40      0.44        10
       Other       0.00      0.00      0.00        10
         Pop       0.04      0.20      0.07        10
         R&B       0.00      0.00      0.00        10
        Rock       0.06      0.10      0.07        10

    accuracy                           0.10       110
   macro avg       0.13      0.10      0.10       110
weighted avg       0.13      0.10      0.10       110

Few-shot medium report
              precision    recall  f1-score   support

     Country       0.75      0.30      0.43        10
  Electronic       0.14      0.

## 15. Running on the full test set

Now, I run the final experiment on the entire test dataset.

In [29]:
zero_shot_full_df = run_experiment(test_df, strategy="zero_shot")
few_shot_full_df = run_experiment(
    test_df,
    strategy="few_shot",
    examples_df=few_shot_examples,
)

100%|██████████| 1500/1500 [2:07:53<00:00,  5.12s/it]


## 16. Full-test evaluation

### 16.1 Printing full reports

In [30]:
print("Zero-shot full report")
print_text_report(zero_shot_full_df, GENRES)

print("Few-shot full report")
print_text_report(few_shot_full_df, GENRES)

Zero-shot full report
              precision    recall  f1-score   support

     Country       0.17      0.30      0.22        89
  Electronic       0.18      0.05      0.08        57
        Folk       0.04      0.29      0.08        17
     Hip-Hop       0.67      0.54      0.60       163
       Indie       0.00      0.00      0.00        16
        Jazz       0.00      0.00      0.00        43
       Metal       0.37      0.44      0.40       151
       Other       0.00      0.00      0.00        31
         Pop       0.24      0.52      0.33       255
         R&B       0.00      0.00      0.00        23
        Rock       0.48      0.21      0.29       655

   micro avg       0.31      0.31      0.31      1500
   macro avg       0.20      0.21      0.18      1500
weighted avg       0.38      0.31      0.31      1500

Few-shot full report
              precision    recall  f1-score   support

     Country       0.49      0.29      0.37        89
  Electronic       0.11      0.09  

### 16.2 Creating per-genre comparison table

In [31]:
zero_metrics_full_df = classification_report_df(zero_shot_full_df, GENRES).rename(
    columns={
        "precision": "precision_zero",
        "recall": "recall_zero",
        "f1": "f1_zero",
    }
)

few_metrics_full_df = classification_report_df(few_shot_full_df, GENRES).rename(
    columns={
        "precision": "precision_few",
        "recall": "recall_few",
        "f1": "f1_few",
    }
)
# merging both tables
comparison_df = zero_metrics_full_df.merge(
    few_metrics_full_df,
    on=["genre", "support"],
)

display(comparison_df)

,genre,precision_zero,recall_zero,f1_zero,support,precision_few,recall_few,f1_few
0,Country,0.170886,0.303371,0.218623,89,0.490566,0.292135,0.366197
1,Electronic,0.176471,0.052632,0.081081,57,0.113636,0.087719,0.099010
2,Folk,0.043478,0.294118,0.075758,17,0.142857,0.352941,0.203390
3,Hip-Hop,0.666667,0.539877,0.596610,163,0.659091,0.533742,0.589831
4,Indie,0.000000,0.000000,0.000000,16,0.026786,0.187500,0.046875
5,Jazz,0.000000,0.000000,0.000000,43,0.428571,0.069767,0.120000
6,Metal,0.370166,0.443709,0.403614,151,0.490741,0.350993,0.409266
7,Other,0.000000,0.000000,0.000000,31,0.055556,0.193548,0.086331
8,Pop,0.236984,0.517647,0.325123,255,0.321526,0.462745,0.379421
9,R&B,0.000000,0.000000,0.000000,23,0.047619,0.043478,0.045455


### 16.3 Overall comparison table

In [32]:
overall_comparison_df = overall_scores_df(zero_shot_full_df, GENRES).merge(
    overall_scores_df(few_shot_full_df, GENRES),
    on="average",
    suffixes=("_zero", "_few"),
)

display(overall_comparison_df)

,average,precision_zero,recall_zero,f1_zero,precision_few,recall_few,f1_few
0,macro,0.195208,0.214870,0.181419,0.310891,0.259034,0.248255
1,weighted,0.378085,0.307333,0.306375,0.505890,0.325333,0.372141


## 17. Inspecting problematic outputs

Here, I check whether any predictions still become `UNKNOWN` after normalization and shows short examples of raw versus normalized outputs.

I also count how many predictions remain `UNKNOWN` after normalization to compare how stable the zero-shot and few-shot outputs are.

In [36]:
print("Number of UNKNOWN predictions")
print("Zero-shot:", (zero_shot_full_df["pred_genre"] == "UNKNOWN").sum())
print("Few-shot:", (few_shot_full_df["pred_genre"] == "UNKNOWN").sum())

Number of UNKNOWN predictions
Zero-shot: 2
Few-shot: 226


In [33]:
print("Zero-shot UNKNOWN predictions:")
display(zero_shot_full_df[zero_shot_full_df["pred_genre"] == "UNKNOWN"].head(10))

print("Few-shot UNKNOWN predictions:")
display(few_shot_full_df[few_shot_full_df["pred_genre"] == "UNKNOWN"].head(10))

Zero-shot UNKNOWN predictions:


,true_genre,raw_output,pred_genre
900,Metal,None,UNKNOWN
933,Metal,None,UNKNOWN


Few-shot UNKNOWN predictions:


,true_genre,raw_output,pred_genre
11,Country,I'll classify this new lyric.\n\nBased on the ...,UNKNOWN
14,Pop,"A fun one!\n\nAfter analyzing the lyrics, I wo...",UNKNOWN
18,Pop,"After analyzing the lyrics, I would classify t...",UNKNOWN
21,Rock,"Based on the lyrics provided, I would classify...",UNKNOWN
35,Pop,"Based on the lyrics provided, I would classify...",UNKNOWN
39,Hip-Hop,"Based on the lyrics, I would classify them as:...",UNKNOWN
46,Metal,"After analyzing the lyrics, I would classify t...",UNKNOWN
62,Rock,"Based on the lyrics provided, I would classify...",UNKNOWN
63,Jazz,"After analyzing the lyrics, I would classify t...",UNKNOWN
71,Rock,"Based on the lyrics, I would classify this son...",UNKNOWN


In [34]:
inspection_zero_df = pd.DataFrame({
    "true_genre": zero_shot_full_df["true_genre"],
    "raw_output": zero_shot_full_df["raw_output"],
    "normalized_prediction": zero_shot_full_df["pred_genre"],
})

inspection_few_df = pd.DataFrame({
    "true_genre": few_shot_full_df["true_genre"],
    "raw_output": few_shot_full_df["raw_output"],
    "normalized_prediction": few_shot_full_df["pred_genre"],
})

print("Example of raw vs normalized zero-shot outputs")
display(inspection_zero_df.head(12))

print("Example of raw vs normalized few-shot outputs")
display(inspection_few_df.head(12))

Example of raw vs normalized zero-shot outputs


,true_genre,raw_output,normalized_prediction
0,Hip-Hop,Metal.,Metal
1,Country,Pop,Pop
2,Rock,Pop,Pop
3,Electronic,Rock,Rock
4,Pop,Country.,Country
5,Indie,Rock.,Rock
6,Rock,Indie,Indie
7,Rock,Rock,Rock
8,Folk,Country,Country
9,Rock,Pop.,Pop


Example of raw vs normalized few-shot outputs


,true_genre,raw_output,normalized_prediction
0,Hip-Hop,"Based on the provided lyrics, I would classify...",Hip-Hop
1,Country,"Based on the provided lyrics, it's difficult t...",Country
2,Rock,"Based on the lyrics provided, I would classify...",Rock
3,Electronic,"Based on the lyrics provided, it is difficult ...",Folk
4,Pop,"After analyzing the lyrics, I would classify t...",Pop
5,Indie,"Based on the lyrics, I would classify this son...",Pop
6,Rock,"Based on the style, tone, and lyrical content,...",Rock
7,Rock,"After analyzing the lyrics, I would classify t...",Metal
8,Folk,"Based on the lyrics, I would classify it as Fo...",Folk
9,Rock,"After analyzing the lyrics, I would classify t...",Pop


## 18. Saving results

The resulting `.csv` files will be temporarily saved inside the Colab Files panel. They can be manually downloaded from there.

In [35]:
zero_shot_full_df.to_csv("zero_shot_full_results.csv", index=False)
few_shot_full_df.to_csv("few_shot_full_results.csv", index=False)
comparison_df.to_csv("genre_comparison_results.csv", index=False)

print("Saved:")
print("- zero_shot_full_results.csv")
print("- few_shot_full_results.csv")
print("- genre_comparison_results.csv")

Saved:
- zero_shot_full_results.csv
- few_shot_full_results.csv
- genre_comparison_results.csv
